In [7]:
from astropy.io import fits
import numpy as np


for fname in [
    "/Users/vdk/data/rcw86/G315.4-2.3_Irefit.fits",
    "/Users/vdk/data/rcw86/G315.42.3_I_CropI.fits",
    "/Users/vdk/data/rcw86/G315.4-2.3_DeepRM_Crop.fits",
    "/Users/vdk/data/rcw86/G315.4-2.3_ShallowRM.fits",
]:
    with fits.open(fname) as hdul:
        hdr0 = hdul[0].header
        print(f"{fname}: NAXIS = {hdr0['NAXIS']}")
        if hdr0['NAXIS'] >= 3:
            print("  CTYPE3 =", hdr0.get('CTYPE3'))
            print("  CRVAL3 =", hdr0.get('CRVAL3'), hdr0.get('CDELT3'), hdr0.get('CRPIX3'))
        print()

/Users/vdk/data/rcw86/G315.4-2.3_Irefit.fits: NAXIS = 4
  CTYPE3 = SPECLNMF
  CRVAL3 = 1335300000.0 57551080.0 1.0

/Users/vdk/data/rcw86/G315.42.3_I_CropI.fits: NAXIS = 4
  CTYPE3 = SPECLNMF
  CRVAL3 = 1335299968.0 57551080.0 1.0

/Users/vdk/data/rcw86/G315.4-2.3_DeepRM_Crop.fits: NAXIS = 4
  CTYPE3 = MaxRMSyn
  CRVAL3 = 299792456578.5 1.0 1.0

/Users/vdk/data/rcw86/G315.4-2.3_ShallowRM.fits: NAXIS = 4
  CTYPE3 = MaxRMSyn
  CRVAL3 = 299792456578.5 1.0 1.0



In [9]:
filename = "/Users/vdk/data/rcw86/G315.4-2.3_Irefit.fits"

hdr = fits.getheader(filename, ext=0)

nchan = hdr["NAXIS3"]      # 5
crval = hdr["CRVAL3"]      # 1.3353000000e9 [Hz]
cdelt = hdr["CDELT3"]      # 5.75108e7    [Hz]
crpix = hdr["CRPIX3"]      # 1.0           (reference pixel)

# compute all channel freqs:
freqs = crval + (np.arange(nchan) + 1 - crpix) * cdelt
freqs

array([1.33530000e+09, 1.39285108e+09, 1.45040216e+09, 1.50795324e+09,
       1.56550432e+09])

In [14]:
data = fits.getdata(filename=filename)
Icube = data[:,0,:,:]     # (nchan, ny, nx), Stokes I = pol index 0
Icube

array([[[           nan,            nan,            nan, ...,
          2.4085243e-04,  9.9251018e-05,  2.0660949e-05],
        [           nan,            nan,            nan, ...,
         -3.1265925e-05, -1.4382768e-04, -1.6732028e-04],
        [           nan,            nan,            nan, ...,
         -2.2531643e-04, -3.1166838e-04, -2.9339266e-04],
        ...,
        [           nan,            nan,            nan, ...,
          3.8229418e-04,  1.3874649e-04, -1.3388139e-04],
        [           nan,            nan,            nan, ...,
          4.3631549e-04,  1.6890865e-04, -1.3120903e-04],
        [           nan,            nan,            nan, ...,
          3.6335716e-04,  9.1712573e-05, -2.0617954e-04]]], dtype='>f4')

In [20]:
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
hdul = fits.open(filename)
hdr  = hdul[0].header
data = hdul[0].data          # shape (nchan, npol, ny, nx)
hdul.close()
hdr.keys

<bound method Header.keys of SIMPLE  =                    T / file does conform to FITS standard             
BITPIX  =                  -32 / number of bits per data pixel                  
NAXIS   =                    4 / number of data axes                            
NAXIS1  =                 5979 / length of data axis 1                          
NAXIS2  =                 5979 / length of data axis 2                          
NAXIS3  =                    5 / length of data axis 3                          
NAXIS4  =                    1 / length of data axis 4                          
EXTEND  =                    T / FITS dataset may contain extensions            
COMMENT   FITS (Flexible Image Transport System) format is defined in 'Astronomy
COMMENT   and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H 
CTYPE1  = 'RA---SIN'           / Axis type                                      
CDELT1  =        -3.344837E-04 / Axis coordinate increment                      

In [28]:
data.shape

(1, 5, 5979, 5979)

In [29]:
# 2. Pick out Stokes I (assumes STOKES is the 4th axis, index 0 → I)
Icube = data[0, :, :, :]     # shape (nchan, ny, nx)

# 3. Build your region mask
#    Example: a circular aperture of radius R around pixel (cx,cy)
ny, nx = Icube.shape[1:]
y, x   = np.indices((ny, nx))
cx, cy, R = nx//2, ny//2, 100   # tweak these to where your region really is
mask = (x - cx)**2 + (y - cy)**2 <= R**2

# 4. Compute the Jy/beam → Jy/pixel conversion factor
#    Beam major/minor axes are in degrees; pixels are in deg per pixel
bmaj = 2.1783e-03            # beam FWHM major axis [deg] BMAJ
bmin = 2.0589e-03          # beam FWHM minor axis [deg] BMIN
pix1 = abs(hdr["CDELT1"])     # deg / pixel in X
pix2 = abs(hdr["CDELT2"])     # deg / pixel in Y
beam_area = np.pi * bmaj * bmin / (4 * np.log(2))   # deg²
pix_area  = pix1 * pix2                              # deg²
conv = pix_area / beam_area                         # Jy/beam → Jy/pix

# 5. Sum up the flux in each channel
nchan = hdr["NAXIS3"]
fluxes = np.zeros(nchan)
for i in range(nchan):
    plane = Icube[i]           # 2D map at channel i
    fluxes[i] = np.sum(plane[mask] * conv)

# 6. Inspect your results
print("Integrated flux per channel [Jy]:")
for i, S in enumerate(fluxes, start=1):
    print(f" Channel {i:1d}: {S:.3f} Jy")

Integrated flux per channel [Jy]:
 Channel 1: -0.008 Jy
 Channel 2: nan Jy
 Channel 3: 0.007 Jy
 Channel 4: 2.471 Jy
 Channel 5: 744.730 Jy


In [27]:
Icube.shape

(1, 5979, 5979)

In [30]:
coeffs = np.polyfit(np.log10(freqs), np.log10(fluxes), 1)
alpha  = coeffs[0]

/var/folders/4h/_cwm20q94xd2kw00z4fz97j80000gn/T/ipykernel_62720/2242267968.py:1: RuntimeWarning: invalid value encountered in log10
  coeffs = np.polyfit(np.log10(freqs), np.log10(fluxes), 1)


In [31]:
alpha

nan